## 📖 BookNetworkAnalyzer 사용법 및 예시 가이드

이 노트북은 소설 텍스트 데이터를 기반으로 **인물 간의 관계망** 및 **특정 인물 중심의 품사별(형용사, 명사, 동사) 연관 단어 네트워크**를 추출하고, interactive한 3D 형태의 **PyVis 네트워크 시각화**를 제공.

---

### 💡 핵심 기능 및 흐름

1. **텍스트 파일 로드 및 문장 분석**
   - 여러 인코딩 방식을 자동으로 시도하며 소설 텍스트를 로딩
2. **정밀 인물 추출**
   - `spaCy` 자연어 처리 모델을 활용해 불용어와 일반 단어를 제거하고 고유명사(PROPN) 위주의 인물 목록을 정밀 필터링.
3. **동시 발생(Co-occurrence) 기반 데이터 빌드**
   - 문장 내에 인물들이 동시에 등장하는 빈도를 측정하여 상호작용 네트워크를 구성.
4. **PyVis 기반 시각화**
   - **전체 인물 관계도**: 기본 상태로는 모든 노드의 크기가 일정하며, 옵션에 따라 중심성(중요도)이 높은 인물의 크기가 커지도록 설정할 수 있음.
   - **인물 연관 단어망**: 특정 인물이 언급된 문장에서 함께 자주 사용된 형용사(초록), 명사(파랑), 동사(노랑)를 분류하여 관계망 그리기.

---

### 🛠️ 실제 사용 예시 코드

소설 데이터 분석은 아래의 코드 참조

```python
# 1. 분석기 객체 생성 및 도서 로드
analyzer = BookNetworkAnalyzer()
analyzer.load_book("/path/to/your/novel.txt")

# 2. 전처리 및 인물 자동 추출
analyzer.get_sentence_sequences()
analyzer.process_characters()

# 3. 네트워크 데이터 빌드 (동시 등장 문장 기준 문턱값 threshold 조정 가능)
analyzer.build_network_data(threshold=2)

# 4. 전체 관계망 시각화 옵션 설정
# apply_importance_size를 True로 변경 시 중심성에 따라 노드 크기가 다르게 표현.
all_char_html = analyzer.plot_network_by_importance_styling(
    centrality_type='pagerank',      # 'degree' (연결수), 'weighted' (가중치), 'pagerank' 선택 가능
    node_color='#3498DB',             # 노드 기본 색상
    base_size=15,                     # 기본 노드 크기
    apply_importance_size=False,      # 중요도에 따른 노드 크기 가변 여부 (기본값: False)
    output_filename='my_network.html'
)
display(all_char_html)

# 5. 특정 인물 기반 연관 단어망 추출
# 해당 인물이 언급된 문장에서 빈출 품사(형용사/명사/동사)를 추출.
word_network_html = analyzer.visualize_character_words(
    target_char="Harry",              # 분석할 대상 인물명
    top_n=10,                         # 품사별 추출할 단어 수
    apply_importance_size=True,       # 함께 쓰인 빈도수에 따라 단어 노드 크기 조절 여부
    output_filename='word_network.html'
)
display(word_network_html)
```

In [ ]:
# Pyvis 설치
!pip install pyvis -q

In [ ]:
# [필수 라이브러리 불러오기 및 한글 글꼴/환경 오류 방지 설정]
import os
import csv
import string
import warnings
from collections import Counter
import nltk
import numpy as np
import pandas as pd
import spacy
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pyvis.network import Network
from IPython.display import HTML, display

warnings.filterwarnings('ignore')

# NLTK 필수 리소스 다운로드
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

In [ ]:
# 1. 한글 깨짐 방지를 위한 나눔고딕 글꼴 설정 및 캐시 삭제
os.system('apt-get install -y fonts-nanum > /dev/null')
os.system('rm -rf ~/.cache/matplotlib')

import matplotlib.font_manager as fm
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = 'NanumBarunGothic'
    plt.rcParams['font.sans-serif'] = ['NanumBarunGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 부호 깨짐 방지

In [ ]:
# NetworkX 그래프 G가 정의되어 있다면 이를 Pyvis로 시각화합니다.
try:
    if 'analyzer' in globals() and analyzer.G is not None:
        net = Network(notebook=True, cdn_resources='in_line')
        net.from_nx(analyzer.G)
        net.show('example.html')

        with open('example.html', 'r', encoding='utf-8') as f:
            html_content = f.read()
        display(HTML(html_content))
    else:
        print("[안내] 아직 빌드된 네트워크(analyzer.G)가 없습니다. 아래의 분석 과정을 마친 후 시각화를 다시 실행해주세요.")
except Exception as e:
    print(f"시각화 중 오류 발생: {e}")

In [ ]:
class BookNetworkAnalyzer:
    """
    소설 텍스트 분석, 네트워크 구조 추출, 시각화 및 어휘 연관관계를 총괄하는 분석 클래스입니다.
    """
    def __init__(self):
        self.book = ""
        self.sequences = []
        self.candidates = []
        self.characters = []
        self.df_connections = None
        self.cooccurence = None
        self.interaction_df = None
        self.G = None
        self.weighted_degree_centrality = {}

        # spaCy 모델 로드
        try:
            self.nlp = spacy.load('en_core_web_sm')
        except OSError:
            os.system('python -m spacy download en_core_web_sm > /dev/null')
            self.nlp = spacy.load('en_core_web_sm')

    # 1. 텍스트 파일 불러오기
    def load_book(self, book_path, lower=False):
        encodings = ['utf-8', 'cp1252', 'latin-1']
        book = None
        for enc in encodings:
            try:
                with open(book_path, encoding=enc) as f:
                    book = f.read()
                break
            except UnicodeDecodeError:
                continue
        if book is None:
            with open(book_path, encoding='utf-8', errors='replace') as f:
                book = f.read()
        self.book = book.lower() if lower else book
        return self.book

    # 2. 문장 토큰화 진행
    def get_sentence_sequences(self):
        detector = nltk.data.load('tokenizers/punkt/english.pickle')
        self.sequences = detector.tokenize(self.book)
        return self.sequences

    # 3. 고유명사(PROPN) 기반 등장인물 자동 추출 필터링
    def process_characters(self, custom_candidates=None):
        if custom_candidates is None:
            remove_punctuation = lambda s: s.translate(str.maketrans('', '', string.punctuation+'’‘'))
            words = [remove_punctuation(p) for p in self.book.split()]
            unique_words = list(set([w for w in words if w]))

            doc = self.nlp(' '.join(unique_words))
            self.candidates = [word.text for word in doc if word.pos_ == 'PROPN' and len(word.text) > 2]
            self.candidates = [c for c in self.candidates if c.istitle()]
            self.candidates = [c for c in self.candidates if not (c.endswith('s') and c[:-1] in self.candidates)]

            stopwords = set(nltk.corpus.stopwords.words('english'))
            self.candidates = list(set([c.title() for c in self.candidates]) - stopwords)
        else:
            self.candidates = custom_candidates

        self.characters = [tuple([character + ' ']) for character in set(self.candidates)]
        print(f"[분석 완료] 최종 자동 추출된 후보 등장인물 수: {len(self.characters)}명")
        return self.characters

    # 4. 캐릭터 정보 파일 입출력
    def save_characters_to_csv(self, save_path):
        with open(save_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerows(self.characters)

    def load_characters(self, characters_path):
        with open(characters_path) as f:
            reader = csv.reader(f)
            self.characters = [tuple(name.lower()+' ' for name in row) for row in reader]
        return self.characters

    # 5. 동시 등장 관계 매트릭스 및 네트워크 구성 (최대 노드수 제어 기능 포함)
    def build_network_data(self, threshold=5, max_nodes=60):
        if any(len(names) > 1 for names in self.characters):
            self.df_connections = pd.DataFrame({
                str(character): {seq: sum([seq.count(name) for name in character]) for seq in self.sequences}
                for character in self.characters
            })
        else:
            char_list = [c[0] for c in self.characters]
            self.df_connections = pd.DataFrame([
                [seq.count(char) for char in char_list] for seq in self.sequences
            ], index=self.sequences, columns=char_list)

        characters_names = self.df_connections.columns.values
        co_matrix = self.df_connections.values.T.dot(self.df_connections.values)
        np.fill_diagonal(co_matrix, 0)
        self.cooccurence = pd.DataFrame(co_matrix, columns=characters_names, index=characters_names)

        rows, columns = np.where(np.triu(self.cooccurence.values, 1) > threshold)
        self.interaction_df = pd.DataFrame(
            np.column_stack([
                self.cooccurence.index[rows],
                self.cooccurence.columns[columns],
                self.cooccurence.values[rows, columns]
            ]),
            columns=['source', 'target', 'value']
        )
        self.interaction_df['value'] = self.interaction_df['value'].astype(int)

        # 시각화 최적화: 중요도가 높은 노드만 필터링
        all_degrees = pd.concat([self.interaction_df['source'], self.interaction_df['target']]).value_counts()
        top_chars = all_degrees.head(max_nodes).index.tolist()

        self.interaction_df = self.interaction_df[
            self.interaction_df['source'].isin(top_chars) & self.interaction_df['target'].isin(top_chars)
        ].reset_index(drop=True)

        self.G = nx.from_pandas_edgelist(self.interaction_df, source='source', target='target')
        G_weighted = nx.from_pandas_edgelist(
            self.interaction_df, source='source', target='target', edge_attr='value', create_using=nx.Graph
        )
        self.weighted_degree_centrality = dict(G_weighted.degree(weight='value'))
        print(f"[네트워크 생성 완료] 노드 {len(self.G.nodes())}개, 엣지 {len(self.G.edges())}개가 생성되었습니다.")
        return self.G

    # 6. PyVis 기반 인터랙티브 인물 관계도 시각화 (Colab Notebook 호환성 수정)
    def plot_network_by_importance_styling(
        self, centrality_type='degree', node_color='#3498DB', base_size=15,
        apply_importance_size=False, scaling_factor=200, output_filename='styled_network.html'
    ):
        if len(self.G.nodes()) == 0:
            print("[알림] 네트워크 데이터가 비어있습니다. build_network_data()를 먼저 실행하십시오.")
            return None

        if centrality_type == 'degree':
            metric_dict = nx.degree_centrality(self.G)
            title_str = "연결 중심성"
        elif centrality_type == 'weighted':
            metric_dict = self.weighted_degree_centrality
            title_str = "가중 연결도"
        elif centrality_type == 'pagerank':
            metric_dict = nx.pagerank(self.G)
            title_str = "PageRank"
        else:
            metric_dict = {node: 1 for node in self.G.nodes()}
            title_str = "일반"

        # Colab 환경을 위해 notebook=True, cdn_resources='in_line' 적용
        net = Network(notebook=True, cdn_resources='in_line', height='650px', width='100%', bgcolor='#222222', font_color='white')

        for node in self.G.nodes():
            val = metric_dict.get(node, 0)
            node_size = (base_size + (val * scaling_factor)) if apply_importance_size else base_size
            net.add_node(node, label=node, title=f"{title_str}: {val:.4f}", color=node_color, size=node_size)

        for u, v in self.G.edges():
            net.add_edge(u, v, color='#888888', alpha=0.5)

        physics_settings = '{"physics": {"barnesHut": {"gravitationalConstant": -4000, "centralGravity": 0.1, "springLength": 90, "springConstant": 0.04}, "stabilization": {"enabled": true, "iterations": 100}}}'
        net.set_options(physics_settings)
        net.show(output_filename)

        with open(output_filename, 'r', encoding='utf-8') as f:
            html_content = f.read()
        return HTML(html_content)

    # 7. 단일 인물의 품사별 연관어(형용사/명사/동사) 네트워크 PyVis 시각화 (Colab Notebook 호환성 수정)
    def visualize_character_words(
        self, target_char, top_n=10, base_size=15, apply_importance_size=False,
        scaling_factor=5, output_filename='character_words_network.html'
    ):
        target_char_lower = target_char.lower().strip()
        matching_sentences = [seq for seq in self.sequences if target_char_lower in seq.lower()]

        if not matching_sentences:
            print(f"[알림] '{target_char}'가 언급된 문장을 찾을 수 없습니다.")
            return None

        words_by_pos = {'ADJ': [], 'NOUN': [], 'VERB': []}
        for sentence in matching_sentences:
            doc = self.nlp(sentence)
            for token in doc:
                if token.text.lower() in target_char_lower or token.is_stop or token.is_punct:
                    continue
                if token.pos_ in ['NOUN', 'PROPN']:
                    words_by_pos['NOUN'].append(token.lemma_.lower())
                elif token.pos_ == 'ADJ':
                    words_by_pos['ADJ'].append(token.lemma_.lower())
                elif token.pos_ == 'VERB':
                    words_by_pos['VERB'].append(token.lemma_.lower())

        # Colab 환경을 위해 notebook=True, cdn_resources='in_line' 적용
        net = Network(notebook=True, cdn_resources='in_line', height='600px', width='100%', bgcolor='#111111', font_color='white')
        net.add_node(target_char, label=target_char, title=f"Target: {target_char}", color='#E74C3C', size=base_size * 1.5)

        pos_colors = {'ADJ': '#2ECC71', 'NOUN': '#3498DB', 'VERB': '#F1C40F'}
        for pos_type, words in words_by_pos.items():
            counts = Counter(words).most_common(top_n)
            for word, freq in counts:
                node_size = (base_size + (freq * scaling_factor)) if apply_importance_size else base_size
                net.add_node(
                    word,
                    label=f"{word} ({pos_type[0]})",
                    title=f"품사: {pos_type}, 빈도: {freq}",
                    color=pos_colors[pos_type],
                    size=node_size
                )
                net.add_edge(target_char, word, value=freq, color='#95A5A6', title=f"출현 빈도: {freq}")

        physics_settings = '{"physics": {"barnesHut": {"gravitationalConstant": -3000, "springLength": 100}, "stabilization": {"enabled": true, "iterations": 100}}}'
        net.set_options(physics_settings)
        net.show(output_filename)

        with open(output_filename, 'r', encoding='utf-8') as f:
            html_content = f.read()
        return HTML(html_content)

    # 8. [신규 통합 기능] 여러 인물의 명사/동사/형용사 연관 단어를 통합하여 단일 파일로 추출 및 저장
    def save_integrated_word_networks(self, target_characters, top_n=15, output_json='integrated_word_networks.json'):
        """
        지정된 인물 목록들의 형용사, 명사, 동사 동시 출현 핵심 단어 관계를 수집하여 하나의 JSON 파일로 저장합니다.
        """
        integrated_data = {}

        for char in target_characters:
            char_clean = char.strip()
            char_lower = char_clean.lower()
            matching_sents = [seq for seq in self.sequences if char_lower in seq.lower()]

            if not matching_sents:
                continue

            words_pos_data = {'ADJ': [], 'NOUN': [], 'VERB': []}
            for sentence in matching_sents:
                doc = self.nlp(sentence)
                for token in doc:
                    if token.text.lower() in char_lower or token.is_stop or token.is_punct:
                        continue
                    if token.pos_ in ['NOUN', 'PROPN']:
                        words_pos_data['NOUN'].append(token.lemma_.lower())
                    elif token.pos_ == 'ADJ':
                        words_pos_data['ADJ'].append(token.lemma_.lower())
                    elif token.pos_ == 'VERB':
                        words_pos_data['VERB'].append(token.lemma_.lower())

            pos_summary = {}
            for pos, words in words_pos_data.items():
                pos_summary[pos] = [{'word': w, 'count': c} for w, c in Counter(words).most_common(top_n)]

            integrated_data[char_clean] = pos_summary

        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(integrated_data, f, ensure_ascii=False, indent=4)

        print(f"[통합 저장 완료] {len(integrated_data)}명의 연관 어휘 관계가 단일 파일에 영구 저장되었습니다: {output_json}")
        return integrated_data

    # 9. [신규 유연한 비교 분석 기능] 지정 인물 대조 또는 최상위 중요도 기반 인물의 어휘 특징 비교분석 표 제공
    def compare_character_lexicon(self, target_list=None, auto_top_n=3, word_count_limit=5):
        """
        지정한 인물들이거나, 지정하지 않은 경우 연결 중심성(Centrality)이 가장 높은 상위 인물들을 자동 선정하여
        그들이 품사별로 어떤 단어와 빈출하여 다르게 얽혀있는지 한눈에 매핑 대조할 수 있는 데이터프레임을 생성해 제공합니다.
        """
        if target_list is None:
            # 연결 중심성이 계산된 기존 네트워크로부터 중요도 순 상위 인물 자동 추천 선택
            cent_dict = nx.degree_centrality(self.G)
            sorted_chars = sorted(cent_dict.items(), key=lambda x: x[1], reverse=True)
            target_list = [item[0].strip() for item in sorted_chars[:auto_top_n]]
            print(f"[자동 비교 가동] 연결 중심성 중요도 내림차순 상위 {auto_top_n}명 자동 선정: {target_list}")
        else:
            print(f"[수동 비교 가동] 분석을 원하는 수동 매핑 인물: {target_list}")

        comparison_results = []
        for char in target_list:
            char_clean = char.strip()
            char_lower = char_clean.lower()
            matching_sents = [seq for seq in self.sequences if char_lower in seq.lower()]

            words_pos_data = {'ADJ': [], 'NOUN': [], 'VERB': []}
            for sentence in matching_sents:
                doc = self.nlp(sentence)
                for token in doc:
                    if token.text.lower() in char_lower or token.is_stop or token.is_punct:
                        continue
                    if token.pos_ in ['NOUN', 'PROPN']:
                        words_pos_data['NOUN'].append(token.lemma_.lower())
                    elif token.pos_ == 'ADJ':
                        words_pos_data['ADJ'].append(token.lemma_.lower())
                    elif token.pos_ == 'VERB':
                        words_pos_data['VERB'].append(token.lemma_.lower())

            row_info = {'인물(Character)': char_clean}
            for pos in ['ADJ', 'NOUN', 'VERB']:
                top_words = [w for w, c in Counter(words_pos_data[pos]).most_common(word_count_limit)]
                row_info[f"{pos} (Top Words)"] = ", ".join(top_words)
            comparison_results.append(row_info)

        df_comparison = pd.DataFrame(comparison_results)
        return df_comparison

    # 10. 기존 일괄 백업 결과 보존 프로세스
    def save_analysis_results(
        self, save_dir='/content', target_person='Harry',
        styled_net_file='styled_network_final.html', target_net_file='harry_words_network.html'
    ):
        import shutil
        import json

        os.makedirs(save_dir, exist_ok=True)
        print(f"[저장 프로세스] 분석 결과 파일을 다음 위치에 저장합니다: {save_dir}\n")

        char_list_path = os.path.join(save_dir, 'extracted_characters.csv')
        self.save_characters_to_csv(char_list_path)
        print(f"1. 인물 목록 저장 완료: {char_list_path} (총 {len(self.characters)}명)")

        if self.interaction_df is not None:
            interaction_path = os.path.join(save_dir, 'character_interactions.csv')
            self.interaction_df.to_csv(interaction_path, index=False, encoding='utf-8-sig')
            print(f"2. 인물 상호작용 가중치 데이터 저장 완료: {interaction_path}")

        target_char_lower = target_person.lower().strip()
        matching_sents = [seq for seq in self.sequences if target_char_lower in seq.lower()]

        if matching_sents:
            words_pos_data = {'ADJ': [], 'NOUN': [], 'VERB': []}
            for sentence in matching_sents:
                doc = self.nlp(sentence)
                for token in doc:
                    if token.text.lower() in target_char_lower or token.is_stop or token.is_punct:
                        continue
                    if token.pos_ in ['NOUN', 'PROPN']:
                        words_pos_data['NOUN'].append(token.lemma_.lower())
                    elif token.pos_ == 'ADJ':
                        words_pos_data['ADJ'].append(token.lemma_.lower())
                    elif token.pos_ == 'VERB':
                        words_pos_data['VERB'].append(token.lemma_.lower())

            word_freq_report = {}
            for pos, words in words_pos_data.items():
                word_freq_report[pos] = Counter(words).most_common(20)

            word_report_path = os.path.join(save_dir, f'{target_person}_word_associations.json')
            with open(word_report_path, 'w', encoding='utf-8') as f:
                json.dump(word_freq_report, f, ensure_ascii=False, indent=4)
            print(f"3. '{target_person}' 중심 품사별 연관어 빈도 데이터 저장 완료: {word_report_path}")

        styled_dest = os.path.join(save_dir, styled_net_file)
        target_dest = os.path.join(save_dir, target_net_file)

        if os.path.exists(styled_net_file):
            shutil.copy(styled_net_file, styled_dest)
            print(f"4. 전체 네트워크 시각화 HTML 복사 완료: {styled_dest}")
        if os.path.exists(target_net_file):
            shutil.copy(target_net_file, target_dest)
            print(f"5. '{target_person}' 연관 단어망 HTML 복사 완료: {target_dest}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. 분석 클래스 객체 생성 및 텍스트 파일 로드
analyzer = BookNetworkAnalyzer()
analyzer.load_book("/content/drive/MyDrive/Classroom/Copy of Harry Potter and the Sorcerer.txt")

# 2. 전처리 및 등장인물 추출
analyzer.get_sentence_sequences()
analyzer.process_characters()  # 자동으로 고유명사 기반 등장인물 추출

# 3. 네트워크 데이터 통합 빌드
analyzer.build_network_data(threshold=2)

# 4. 전체 인물 관계망 PyVis 시각화 (기본 노드 크기는 중요도 미반영, apply_importance_size=False)

print("\n--- [전체 인물 관계 네트워크 시각화] ---")
all_char_html = analyzer.plot_network_by_importance_styling(
    centrality_type='pagerank',
    node_color='#3498DB',
    apply_importance_size=False,
    output_filename='styled_network_final.html'
)
display(all_char_html)

# 5. 특정 인물 지정 연관 단어(형용사, 명사, 동사) 네트워크 추출 및 시각화
target_person = "Harry"
print(f"\n--- [특정 인물 '{target_person}' 중심 연관 단어 네트워크 시각화] ---")
word_network_html = analyzer.visualize_character_words(
    target_char=target_person,
    top_n=10,
    apply_importance_size=False,
    output_filename='harry_words_network.html'
)
if word_network_html:
    display(word_network_html)

### 🌟 11. 다중 인물 통합 어휘 분석 및 비교 분석 기법 가동

클래스 통합 구조에 새롭게 적용된 `save_integrated_word_networks()`와 `compare_character_lexicon()` 메소드를 사용하여 주요 인물들을 대조 분석합니다.

### 🚀 6. 클래스 통합 메소드를 통한 데이터 및 시각화 결과 일괄 저장

방금 분석 및 시각화한 결과를 클래스 내부에 새롭게 빌드된 `save_analysis_results()` 통합 메소드를 사용하여 구글 드라이브(혹은 로컬 디렉토리)에 일괄 저장합니다.

In [ ]:
# 저장할 디렉토리 경로 지정 및 일괄 저장 실행
save_directory = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content"

# 클래스 내부의 통합 결과 일괄 저장 메소드 호출
analyzer.save_analysis_results(
    save_dir=save_directory,
    target_person="Harry",
    styled_net_file="styled_network_final.html",
    target_net_file="harry_words_network.html"
)

### 📊 7. 저장된 'Harry' 연관 단어 빈도 분석 결과 확인

이전 단계에서 저장된 `Harry_word_associations.json` 파일을 로드하여, 해리(Harry)와 함께 가장 많이 쓰인 품사별(형용사, 명사, 동사) 상위 단어 빈도 분석 결과를 표 형식으로 직접 확인해 봅니다.

In [ ]:
import json
import os
import pandas as pd

# 통합으로 빌드 및 백업된 Harry_word_associations.json 파일 경로 확인
json_path = os.path.join(save_directory, 'Harry_word_associations.json')

if os.path.exists(json_path):
    # JSON 파일 데이터 읽기
    with open(json_path, 'r', encoding='utf-8') as f:
        associations_data = json.load(f)

    # 품사별 핵심 연관 단어 추출 및 판다스 데이터프레임 형식으로 정갈하게 출력
    for pos, data in associations_data.items():
        pos_name = '형용사 (ADJ)' if pos == 'ADJ' else '명사 (NOUN)' if pos == 'NOUN' else '동사 (VERB)'
        print(f"\n📢 [{pos_name}] 핵심 연관 단어 Top 10")
        df_pos = pd.DataFrame(data, columns=['단어(Word)', '빈도수(Frequency)'])
        display(df_pos.head(10))
else:
    print(f"❌ [{json_path}] 분석 결과 파일이 존재하지 않습니다. 저장이 정상 처리되었는지 점검바랍니다.")

In [ ]:
# 1. 최신 클래스 정의가 반영된 상태로 인스턴스 갱신 및 기존 텍스트 데이터 보존 처리
# (클래스가 변경된 후 인스턴스가 예전 버전에 머무르는 현상 방지)
current_book_path = "/content/drive/MyDrive/Classroom/Copy of Harry Potter and the Sorcerer.txt"

# 신규 인스턴스로 새로 구축
analyzer = BookNetworkAnalyzer()
analyzer.load_book(current_book_path)
analyzer.get_sentence_sequences()
analyzer.process_characters()
analyzer.build_network_data(threshold=2)

# 2. 연결 중심성(Centrality) 상위 5명 인물 리스트 자동 도출 및 통합 저장 실행
top_characters = [char.strip() for char, _ in sorted(nx.degree_centrality(analyzer.G).items(), key=lambda x: x[1], reverse=True)[:5]]
print(f"🎯 자동 감지된 분석 대상 주요 인물 목록: {top_characters}")

# 품사별 어휘 네트워크 단일 통합 JSON 파일 저장
integrated_json_path = os.path.join(save_directory, 'integrated_word_networks.json')
analyzer.save_integrated_word_networks(
    target_characters=top_characters,
    top_n=15,
    output_json=integrated_json_path
)

# 3. 지정 인물 혹은 상위 3명 인물 간의 품사별 사용 어휘 직접 대조 및 가시화
print(
    "\n📊 [비교 대조 분석] 연결망 최상위 인물들의 주요 품사별 빈출 어휘 비교 표"
)
df_lexicon_comparison = analyzer.compare_character_lexicon(
    target_list=top_characters[:3],
    word_count_limit=5
)
display(df_lexicon_comparison)

### 📊 8. 연결 중심성(Degree Centrality)과 원문 등장 빈도 간의 상관관계 분석

네트워크 연결 중심성이 높은 인물이 실제로 본문에도 자주 빈출하는지 확인하기 위해 두 지표 간의 상관관계를 통계적으로 구하고 산점도로 시각화해 봅니다.

In [ ]:
# 1. 네트워크의 연결 중심성 계산
cent_dict = nx.degree_centrality(analyzer.G)

# 2. 각 인물의 텍스트 등장 빈도수 계산
# analyzer.df_connections 에는 문장별 인물의 등장 횟수가 기록되어 있으므로 열 합계를 통해 빈도수 획득
char_frequencies = {}
for char in analyzer.G.nodes():
    if char in analyzer.df_connections.columns:
        char_frequencies[char] = analyzer.df_connections[char].sum()
    else:
        # 공백 문자 등이 포함된 대체 이름 매핑 시도
        alt_name = char.strip().lower() + ' '
        matched = [col for col in analyzer.df_connections.columns if col.strip().lower() == char.strip().lower()]
        if matched:
            char_frequencies[char] = analyzer.df_connections[matched[0]].sum()
        else:
            char_frequencies[char] = 0

# 3. 데이터 통합용 DataFrame 생성
analysis_data = []
for char in analyzer.G.nodes():
    analysis_data.append({
        'Character': char.strip(),
        'Degree Centrality': cent_dict.get(char, 0),
        'Occurrence Frequency': char_frequencies.get(char, 0)
    })

df_corr = pd.DataFrame(analysis_data)

# 4. 상관계수 계산
pearson_corr = df_corr['Degree Centrality'].corr(df_corr['Occurrence Frequency'])
print(f"📌 [상관분석] 연결 중심성과 등장 빈도 간의 피어슨 상관계수: {pearson_corr:.4f}")
if pearson_corr >= 0.7:
    print("👉 해석: 두 변수 간에 강한 양의 상관관계가 존재합니다. 즉, 텍스트에 자주 나올수록 더 복잡한 관계망 중심에 서 있습니다.")
else:
    print("👉 해석: 연결 중심성과 단순 빈도 간에 무난하거나 낮은 상관관계가 관찰됩니다.")

# 5. 상관관계 데이터 프레임 상위 10명 출력
display(df_corr.sort_values(by='Degree Centrality', ascending=False).head(10))

# 6. 시각화 (산점도 및 회귀선 추세선)
plt.figure(figsize=(10, 6))
sns.regplot(data=df_corr, x='Occurrence Frequency', y='Degree Centrality',
            scatter_kws={'s': 80, 'color': '#3498DB', 'alpha': 0.7},
            line_kws={'color': '#E74C3C', 'linewidth': 2})

# 상위 8개 주요 인물 텍스트 라벨 추가
for i, row in df_corr.sort_values(by='Degree Centrality', ascending=False).head(8).iterrows():
    plt.text(row['Occurrence Frequency'] + 5, row['Degree Centrality'], row['Character'],
             fontsize=10, weight='bold', alpha=0.9)

plt.title('등장 빈도수와 연결 중심성의 상관관계 (Correlation Analysis)', fontsize=14, pad=15)
plt.xlabel('본문 내 등장 빈도 (Occurrence Frequency)', fontsize=12)
plt.ylabel('연결 중심성 (Degree Centrality)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()